# Logistic Regression
### Both models are written from scratch in PyTorch in `self_module/logistic.py` (the maths is in the comments there).

In [1]:
import sys
import numpy as np
import pandas as pd
import torch
import plotly.express as px
from sklearn.metrics import log_loss , roc_auc_score , average_precision_score , accuracy_score , precision_score , recall_score , f1_score , confusion_matrix

sys.path.append('../..')
from self_module import LogisticRegression , WeightedLogisticRegression

In [2]:
import plotly.io as pio
pio.templates["plotly_dark"].layout.paper_bgcolor = "black"
pio.templates["plotly_dark"].layout.plot_bgcolor = "black"
pio.templates.default = "plotly_dark"

## 3. Machine Learning
### Why logistic regression : it turns a weighted sum of the features into a click probability with the sigmoid.
### A probability is what an ad platform needs : expected value of an impression = P(click) x price per click.
### Limitations : straight line boundary , so combinations (this app on this device) are missed unless added by hand.
### Two versions : plain , and class weighted (each class carries half the loss , like the weighted SVM).

### Loading the data and adding a column of 1s for the bias. The GPU is used if there is one.

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def load(name):
    X = torch.tensor(pd.read_parquet(f'../../data/ctr/X_{name}.parquet').values , dtype=torch.float32)
    y = torch.tensor(pd.read_parquet(f'../../data/ctr/y_{name}.parquet').values , dtype=torch.float32)
    X = torch.cat([X , torch.ones((len(X) , 1))] , dim=1)
    return X.to(device) , y.to(device)

X_train , y_train = load('train')
X_val , y_val = load('val')
X_test , y_test = load('test')
X_train.shape

torch.Size([1010725, 704])

### Plain logistic regression. max_lr gives the largest learning rate that is still safe.

In [4]:
model = LogisticRegression()
lr = model.max_lr(X_train)
model = LogisticRegression(lr=lr , epochs=2000)
model.train(X_train , y_train , X_val , y_val , log_every=500) ;

Epoch 0 | Loss: 0.6931 | Val loss: 0.5947


Epoch 500 | Loss: 0.4070 | Val loss: 0.4094


Epoch 1000 | Loss: 0.4068 | Val loss: 0.4093


Epoch 1500 | Loss: 0.4067 | Val loss: 0.4092


In [5]:
loss = pd.DataFrame(model.history , columns=['train' , 'val'])
fig = px.line(loss , title='Loss')
fig.show()

### Train and validation loss stay together (0.407 vs 0.409) , so it is not overfitting. Almost all of the drop happens in the first 500 epochs.

### Class weighted logistic regression. max_lr needs the labels too , since the weights come from them.

In [6]:
wmodel = WeightedLogisticRegression()
lr = wmodel.max_lr(X_train , y_train)
wmodel = WeightedLogisticRegression(lr=lr , epochs=2000)
wmodel.train(X_train , y_train , X_val , y_val , log_every=500) ;

Epoch 0 | Loss: 0.6931 | Val loss: 0.6540


Epoch 500 | Loss: 0.6061 | Val loss: 0.6093


Epoch 1000 | Loss: 0.6057 | Val loss: 0.6090


Epoch 1500 | Loss: 0.6056 | Val loss: 0.6089


## Evaluation
### Baseline : always predict the train CTR. Its log loss is what a model has to beat.
### Threshold for precision / recall / F1 : picked on validation by best F1. Log loss , ROC-AUC and PR-AUC don't need one.

In [7]:
def probs(m , X):
    return m.forward(X).cpu().numpy().ravel()

y_val_np , y_test_np = y_val.cpu().numpy().ravel() , y_test.cpu().numpy().ravel()
p_val , p_test = probs(model , X_val) , probs(model , X_test)
pw_val , pw_test = probs(wmodel , X_val) , probs(wmodel , X_test)

def best_threshold(y , p):
    thresholds = np.arange(0.05 , 0.95 , 0.01)
    f1s = [f1_score(y , p >= t) for t in thresholds]
    return thresholds[np.argmax(f1s)]

threshold , wthreshold = best_threshold(y_val_np , p_val) , best_threshold(y_val_np , pw_val)
threshold , wthreshold

(np.float64(0.19), np.float64(0.5300000000000001))

In [8]:
def scores(y , p , pred):
    return {'log loss' : log_loss(y , p) , 'roc_auc' : roc_auc_score(y , p) , 'pr_auc' : average_precision_score(y , p) ,
            'accuracy' : accuracy_score(y , pred) , 'precision' : precision_score(y , pred , zero_division=0) ,
            'recall' : recall_score(y , pred) , 'f1' : f1_score(y , pred)}

base = np.full(len(y_test_np) , y_train.mean().item())
p_train = probs(model , X_train)
y_train_np = y_train.cpu().numpy().ravel()

pd.DataFrame({
    'baseline (test)' : scores(y_test_np , base , base >= 0.5) ,
    'plain (train)'   : scores(y_train_np , p_train , p_train >= threshold) ,
    'plain (test)'    : scores(y_test_np , p_test , p_test >= threshold) ,
    'weighted (test)' : scores(y_test_np , pw_test , pw_test >= wthreshold) ,
}).round(3)

,baseline (test),plain (train),plain (test),weighted (test)
log loss,0.456,0.407,0.409,0.611
roc_auc,0.500,0.728,0.725,0.724
pr_auc,0.170,0.348,0.347,0.345
accuracy,0.830,0.656,0.655,0.651
precision,0.000,0.285,0.285,0.283
recall,0.000,0.683,0.679,0.683
f1,0.000,0.402,0.401,0.400


### Log loss 0.409 vs 0.456 for the baseline , 10% lower. ROC-AUC 0.725. PR-AUC 0.347 , twice the 0.17 a random guess gets.
### Accuracy falls below the baseline's 0.83 on purpose : the F1 threshold (0.19) calls a lot of impressions a click , recall 68% , precision 29%.
### Train and test are almost the same , so the model generalises.
### The weighted model ranks the same (ROC-AUC 0.724) but its log loss is far worse (0.611). The calibration plot below shows why.

In [9]:
cm = confusion_matrix(y_test_np , p_test >= threshold)
fig = px.imshow(cm , text_auto=True , x=['predicted no click' , 'predicted click'] , y=['actual no click' , 'actual click'] , title='Confusion matrix , plain (test)')
fig.show()

### 14,607 clicks caught , 6,903 missed , 36,688 impressions flagged that were not clicked.

### Calibration : when the model says 20% , do 20% of those impressions get clicked ?
### Test rows are cut into 10 bins by predicted probability , then the mean prediction is compared with the real CTR of each bin. On the dashed line = calibrated.

In [10]:
def calibration(p , name):
    bins = pd.qcut(p , 10 , duplicates='drop')
    g = pd.DataFrame({'predicted' : p , 'actual' : y_test_np}).groupby(bins , observed=True).mean()
    g['model'] = name
    return g

pc_test = wmodel.forward_unweighted(X_test).cpu().numpy().ravel()
cal = pd.concat([calibration(p_test , 'plain') , calibration(pw_test , 'weighted') , calibration(pc_test , 'weighted , bias corrected')])
fig = px.line(cal , x='predicted' , y='actual' , color='model' , markers=True , title='Calibration (test)')
fig.add_shape(type='line' , x0=0 , y0=0 , x1=1 , y1=1 , line_dash='dash')
fig.show()

In [11]:
pd.DataFrame({
    'plain'                    : {'mean p' : p_test.mean() , 'log loss' : log_loss(y_test_np , p_test) , 'roc_auc' : roc_auc_score(y_test_np , p_test)} ,
    'weighted'                 : {'mean p' : pw_test.mean() , 'log loss' : log_loss(y_test_np , pw_test) , 'roc_auc' : roc_auc_score(y_test_np , pw_test)} ,
    'weighted , bias corrected' : {'mean p' : pc_test.mean() , 'log loss' : log_loss(y_test_np , pc_test) , 'roc_auc' : roc_auc_score(y_test_np , pc_test)} ,
}).round(4)

,plain,weighted,"weighted , bias corrected"
mean p,0.1699,0.4464,0.1699
log loss,0.4087,0.6108,0.4090
roc_auc,0.7247,0.7242,0.7242


### Plain : on the line. The 10% it rates highest (41% predicted) really click at 40% , the lowest 10% (2% predicted) at 2.5%.
### Weighted : same order , but every prediction is too high (mean 0.45 against a real CTR of 0.17). The weights act like copying every click about 4.9 times.
### Taking log(c_pos / c_neg) off the bias (forward_unweighted in self_module) puts it back on the line , with the same log loss as the plain model.
### For bidding the probability itself is used (P(click) x price) , so the plain model is the one to keep. The weights don't even raise F1 here.

### Simple sklearn implementation

In [12]:
from sklearn.linear_model import LogisticRegression as SkLogisticRegression

sk_model = SkLogisticRegression(C=1e12 , max_iter=1000)
sk_model.fit(pd.read_parquet('../../data/ctr/X_train.parquet') , y_train_np)
p_sk = sk_model.predict_proba(pd.read_parquet('../../data/ctr/X_test.parquet'))[: , 1]

### Our model vs sklearn

In [13]:
pd.DataFrame({
    'our model' : scores(y_test_np , p_test , p_test >= threshold) ,
    'sklearn'   : scores(y_test_np , p_sk , p_sk >= threshold) ,
}).round(3)

,our model,sklearn
log loss,0.409,0.409
roc_auc,0.725,0.725
pr_auc,0.347,0.347
accuracy,0.655,0.655
precision,0.285,0.285
recall,0.679,0.679
f1,0.401,0.402


### Both give the same results.

## 4. Inference and Interpretation
### Feature importance : the features are standardised , so each weight is the change in log odds for 1 standard deviation of that column.
### Positive pushes towards a click , negative away. The 15 largest each way :

In [14]:
features = pd.read_parquet('../../data/ctr/X_train.parquet').columns
weights = pd.Series(model.W[:-1].cpu().numpy().ravel() , index=features).sort_values()
top = pd.concat([weights.head(15) , weights.tail(15)])
fig = px.bar(top , orientation='h' , height=800 , title='Weights (15 most negative , 15 most positive)')
fig.show()

### Towards a click : site_id 85f751fd (the in-app placeholder) , banner position 1 , some sites (d9750ee7 , 5ee41ff2) and some C21 / C17 levels.
### Away from a click : the two biggest site categories (50e219e0 , f028772b) and the low CTR app categories (cef3e649 , 0f2161f8 , 8ded1f7a) , like the CTR bars in EDA.
### Many columns overlap (a site_id sits inside one site_category) , so each weight is the effect with the others held fixed. That is why site category 3e814130 has 28% CTR in EDA but a negative weight here : its sites already carry it.
### These show which features go together with a click , not what causes it.

In [15]:
# saved for the comparison in 04_svm
pd.DataFrame({'p' : p_test , 'pred' : (p_test >= threshold).astype(int) , 'pw' : pw_test , 'pred_w' : (pw_test >= wthreshold).astype(int)}).to_parquet('../../data/ctr/logistic_test.parquet')